# Notebook 1 — Current Snapshot EDA

Exploratory analysis of the **current** fraud + value metrics report.

Each row = one company (latest annual 10-K, current market price).  
Proxy ROI = `price_change_90d` (recent 90-day price change, already in data).  
Full historical ROI is in Notebook 2 (multi-year dataset).

**Sections:**
1. Dataset shape & coverage  
2. Market cap segment distribution  
3. Proxy ROI (`price_change_90d`) distribution  
4. Volatility, Beta, Bid/Ask spread  
5. Fraud score distribution by segment  
6. Signal vs proxy ROI correlations  
7. Magic Formula top candidates  
8. Alpha screen: cheap + quality + low fraud  
9. Net-net companies (Graham floor)  
10. Full signal correlation heatmap

## Setup — run this first (works in Google Colab and locally)

In [ ]:
import json, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('dark_background')

# ── Detect environment and load data ─────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    import subprocess
    subprocess.run([
        'wget', '-q', '-O', 'fraud_report.json',
        'https://raw.githubusercontent.com/sherlock718/stock-fraud-screener/main/reports/fraud_report.json'
    ])
    report_path = 'fraud_report.json'
else:
    report_path = os.path.join(os.path.dirname(os.getcwd()), 'reports', 'fraud_report.json')
    if not os.path.exists(report_path):
        report_path = '../reports/fraud_report.json'

with open(report_path) as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Loaded {len(df)} companies | {len(df.columns)} columns")
df.head(3)

## 1. Dataset shape & coverage

In [ ]:
# Coverage report for all numeric columns
numeric_cols = df.select_dtypes(include=[float, int]).columns
coverage = (
    df[numeric_cols].notna().sum()
    .sort_values(ascending=False)
    .reset_index()
)
coverage.columns = ['metric', 'count']
coverage['coverage_%'] = (coverage['count'] / len(df) * 100).round(1)

# Plot top-40 by coverage
top40 = coverage.head(40)
fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#2ecc71' if x >= 70 else '#f39c12' if x >= 40 else '#e74c3c' for x in top40['coverage_%']]
ax.barh(top40['metric'][::-1], top40['coverage_%'][::-1], color=colors[::-1])
ax.axvline(70, color='red', linestyle='--', alpha=0.5, label='70% threshold')
ax.set_xlabel('Coverage (%)')
ax.set_title('Feature Coverage in Current Report')
ax.legend()
plt.tight_layout()
plt.show()
print(coverage.to_string(index=False))

## 2. Market cap segment distribution

In [ ]:
seg_order = ['micro', 'small', 'mid', 'large']
seg_counts = df['market_cap_segment'].value_counts().reindex(seg_order).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Count
axes[0].bar(seg_order, seg_counts, color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'])
axes[0].set_title('Companies by Market Cap Segment')
axes[0].set_ylabel('Count')

# Pie
axes[1].pie(seg_counts, labels=seg_order, autopct='%1.1f%%',
             colors=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'])
axes[1].set_title('Market Cap Composition')
plt.tight_layout()
plt.show()
print(seg_counts.to_string())

## 3. Proxy ROI — `price_change_90d`

Since we don't yet have the full historical dataset, we use **90-day price change** as a proxy  
short-term return signal. Full 12-month forward returns are in Notebook 2.

In [ ]:
roi = df['price_change_90d'].dropna()
roi_clipped = roi.clip(-0.8, 2)  # clip extreme outliers for display

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
axes[0].hist(roi_clipped, bins=80, color='#3498db', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='white', linestyle='--', alpha=0.5)
axes[0].axvline(roi.median(), color='#2ecc71', linestyle='-', label=f'Median {roi.median():.1%}')
axes[0].set_title('90-Day Price Change Distribution')
axes[0].set_xlabel('Return (clipped at -80% / +200%)')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].legend()

# By segment
roi_by_seg = df.groupby('market_cap_segment')['price_change_90d'].median().reindex(seg_order)
axes[1].bar(seg_order, roi_by_seg, color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'])
axes[1].axhline(0, color='white', linestyle='--', alpha=0.5)
axes[1].set_title('Median 90d Return by Segment')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# Positive vs negative
pct_pos = (roi > 0).mean()
axes[2].pie([pct_pos, 1-pct_pos], labels=[f'Positive ({pct_pos:.1%})', f'Negative ({1-pct_pos:.1%})'],
             colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%')
axes[2].set_title('90d Return Positive vs Negative')

plt.tight_layout()
plt.show()

print(f"Companies with 90d data: {len(roi):,} / {len(df):,}")
print(f"Mean:   {roi.mean():.1%}")
print(f"Median: {roi.median():.1%}")
print(f"Std:    {roi.std():.1%}")
print(f">0:     {(roi > 0).mean():.1%} of companies had positive 90d return")

## 4. Volatility, Beta & Bid/Ask Spread

Risk and liquidity profile of the universe.

In [ ]:
risk_cols = ['volatility_90d', 'beta', 'bid_ask_spread']
available = [c for c in risk_cols if c in df.columns and df[c].notna().sum() > 10]

if not available:
    print("Volatility/beta data not yet populated — run enrich_market_signals.py first")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(6*len(available), 4))
    if len(available) == 1:
        axes = [axes]

    labels = {'volatility_90d': 'Annualised Volatility (90d)',
               'beta': 'Beta vs S&P 500',
               'bid_ask_spread': 'Bid/Ask Spread'}
    colors_map = {'volatility_90d': '#e74c3c', 'beta': '#3498db', 'bid_ask_spread': '#f39c12'}

    for ax, col in zip(axes, available):
        data_col = df[col].dropna()
        data_col_clipped = data_col.clip(data_col.quantile(0.01), data_col.quantile(0.99))
        ax.hist(data_col_clipped, bins=60, color=colors_map[col], edgecolor='none', alpha=0.8)
        ax.axvline(data_col.median(), color='white', linestyle='--',
                    label=f'Median {data_col.median():.3f}')
        ax.set_title(labels[col])
        ax.set_xlabel(col)
        ax.legend()
        ax.set_ylabel('Count')

    plt.suptitle('Market Risk & Liquidity Metrics', y=1.02)
    plt.tight_layout()
    plt.show()

    # By segment
    if 'volatility_90d' in available:
        vol_by_seg = df.groupby('market_cap_segment')['volatility_90d'].median().reindex(seg_order)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(seg_order, vol_by_seg, color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'])
        ax.set_title('Median Annualised Volatility by Market Cap Segment')
        ax.set_ylabel('Annualised Vol')
        plt.tight_layout()
        plt.show()

## 5. Fraud score distribution by segment

Key question: do small/micro caps have higher fraud scores on average?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = {'micro': '#e74c3c', 'small': '#f39c12', 'mid': '#3498db', 'large': '#2ecc71'}

for ax, seg in zip(axes, seg_order):
    subset = df[df['market_cap_segment'] == seg]['fraud_score'].dropna()
    ax.hist(subset, bins=30, color=colors[seg], alpha=0.8, edgecolor='none')
    ax.set_title(f'{seg.title()} (n={len(subset):,})')
    ax.set_xlabel('Fraud Score')
    ax.axvline(70, color='red', linestyle='--', alpha=0.5, label='High risk (70)')
    ax.axvline(45, color='orange', linestyle='--', alpha=0.5, label='Medium (45)')
    ax.axvline(subset.median(), color='white', linewidth=2,
                label=f'Median {subset.median():.0f}')
    ax.legend(fontsize=7)

axes[0].set_ylabel('Count')
plt.suptitle('Fraud Score Distribution by Market Cap Segment', y=1.02)
plt.tight_layout()
plt.show()

# Summary stats
seg_stats = df.groupby('market_cap_segment')['fraud_score'].agg(['mean','median','std','count'])
seg_stats = seg_stats.reindex(seg_order)
print(seg_stats.round(1).to_string())

## 6. Signal vs Proxy ROI — correlations

Which signals are most correlated with 90-day price change?  
Note: this is a noisy proxy — 12-month forward return correlations are in Notebook 2.

In [ ]:
label = 'price_change_90d'
roi_df = df[df[label].notna()].copy()
roi_df[label] = roi_df[label].clip(-0.8, 2)

feature_cols = [
    'fraud_score', 'beneish_score', 'piotroski_score', 'accruals_ratio', 'cfd_ratio',
    'altman_score', 'ar_ratio', 'non_op_ratio',
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'croic', 'roe', 'roa', 'fcf_yield',
    'gross_margin', 'net_margin', 'debt_to_equity', 'current_ratio',
    'pe_ratio', 'pb_ratio', 'ev_ebitda', 'ncav_ratio',
    'volatility_90d', 'beta', 'bid_ask_spread', 'magic_formula_rank',
]
existing = [c for c in feature_cols if c in roi_df.columns]

corrs = roi_df[existing + [label]].corr()[label].drop(label).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
c_colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=c_colors, alpha=0.8)
ax.axvline(0, color='white', alpha=0.3)
ax.set_title(f'Pearson Correlation with 90d Price Change (n={len(roi_df):,})')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()

print("Top positive correlations (features that predict higher returns):")
print(corrs.tail(10).to_string())
print("\nTop negative correlations (features that predict lower returns):")
print(corrs.head(10).to_string())

## 6b. Fraud score vs proxy ROI — quartile analysis

In [ ]:
q_df = df[df['price_change_90d'].notna() & df['fraud_score'].notna()].copy()
q_df['fraud_quartile'] = pd.qcut(q_df['fraud_score'], q=4,
                                   labels=['Q1 Low Risk', 'Q2', 'Q3', 'Q4 High Risk'])

avg_ret = q_df.groupby('fraud_quartile', observed=True)['price_change_90d'].agg(['mean', 'median', 'count'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(avg_ret.index, avg_ret['median'],
        color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'], alpha=0.85)
ax.axhline(0, color='white', linestyle='--', alpha=0.4)
ax.set_title('Median 90d Return by Fraud Score Quartile')
ax.set_xlabel('Fraud Score Quartile')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()
print(avg_ret.round(4))

## 7. Magic Formula — top candidates

Low MF rank + low fraud score = the sweet spot (cheap + quality + clean)

In [ ]:
mf = df[df['magic_formula_rank'].notna()].copy()
mf = mf.sort_values('magic_formula_rank')

cols = ['ticker', 'name', 'market_cap_segment', 'fraud_score', 'risk',
        'magic_formula_rank', 'earnings_yield', 'return_on_capital',
        'acquirers_multiple', 'gross_profitability']
cols_exist = [c for c in cols if c in mf.columns]

print(f"Companies with MF rank: {len(mf):,}")
display = mf[cols_exist].head(20).copy()
for c in ['earnings_yield', 'return_on_capital', 'gross_profitability']:
    if c in display:
        display[c] = display[c].apply(lambda x: f'{x:.1%}' if pd.notna(x) else '')
display

## 8. Alpha screen: Cheap + Survive + Low Fraud

Intersection of: top MF rank quartile + low fraud score + not distress zone + no going concern

In [ ]:
mf_threshold = df['magic_formula_rank'].quantile(0.25)  # top 25%

candidates = df[
    (df['magic_formula_rank'] <= mf_threshold) &
    (df['fraud_score'] < 45) &
    (df['altman_zone'].isin(['safe', 'grey'])) &
    (df['going_concern_flag'] == False)
].copy()

candidates = candidates.sort_values('magic_formula_rank')
print(f"Alpha candidates: {len(candidates)}")
print(f"By segment:")
print(candidates['market_cap_segment'].value_counts().reindex(seg_order).to_string())

show_cols = [c for c in cols if c in candidates.columns]
candidates[show_cols].head(30)

### 8b. Enhanced alpha screen — add volatility & quality filters

In [ ]:
has_vol = df['volatility_90d'].notna().sum() > 0

if has_vol:
    vol_threshold = df['volatility_90d'].quantile(0.5)  # below median volatility
    gp_threshold  = df['gross_profitability'].quantile(0.5)  # above median quality

    alpha_enhanced = df[
        (df['magic_formula_rank'] <= mf_threshold) &
        (df['fraud_score'] < 45) &
        (df['altman_zone'].isin(['safe', 'grey'])) &
        (df['going_concern_flag'] == False) &
        (df['volatility_90d'] <= vol_threshold) &        # lower volatility
        (df['gross_profitability'] >= gp_threshold)      # higher quality
    ].copy()

    alpha_enhanced = alpha_enhanced.sort_values('magic_formula_rank')
    print(f"Enhanced alpha candidates (with vol/quality filter): {len(alpha_enhanced)}")

    extra_cols = [c for c in cols + ['volatility_90d', 'gross_profitability'] if c in alpha_enhanced.columns]
    alpha_enhanced[extra_cols].head(20)
else:
    print("Volatility data not yet populated — run enrich_market_signals.py to enable this screen")

## 9. Net-net companies (Graham liquidation floor)

Market cap < NCAV = trading below liquidation value

In [ ]:
net_nets = df[df['net_net_flag'] == True].copy()
net_nets = net_nets.sort_values('ncav_ratio')
print(f"Net-net companies: {len(net_nets)}")
print(f"By segment: {net_nets['market_cap_segment'].value_counts().to_dict()}")

nn_cols = ['ticker', 'name', 'market_cap_segment', 'fraud_score', 'risk',
           'ncav_ratio', 'ncav', 'market_cap', 'current_ratio', 'going_concern_flag']
nn_cols_exist = [c for c in nn_cols if c in net_nets.columns]
net_nets[nn_cols_exist].head(20)

## 10. Full signal correlation heatmap

Which signals move together? High correlation = redundancy.

In [ ]:
signal_cols = [
    'fraud_score', 'beneish_score', 'piotroski_score', 'accruals_ratio', 'cfd_ratio',
    'altman_score', 'ar_ratio', 'non_op_ratio',
    'earnings_yield', 'return_on_capital', 'acquirers_multiple',
    'gross_profitability', 'croic', 'roe', 'roa', 'fcf_yield',
    'gross_margin', 'net_margin', 'debt_to_equity', 'current_ratio',
    'pe_ratio', 'pb_ratio', 'ev_ebitda', 'ncav_ratio',
    'volatility_90d', 'beta', 'bid_ask_spread', 'price_change_90d',
]
existing = [c for c in signal_cols if c in df.columns and df[c].notna().sum() > 100]
corr = df[existing].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(existing)))
ax.set_yticks(range(len(existing)))
ax.set_xticklabels(existing, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(existing, fontsize=9)
ax.set_title('Signal Correlation Matrix')
plt.tight_layout()
plt.show()

# Pairs with high correlation (redundant signals)
import itertools
high_corr = []
for a, b in itertools.combinations(existing, 2):
    v = corr.loc[a, b]
    if abs(v) > 0.6:
        high_corr.append({'feature_a': a, 'feature_b': b, 'correlation': round(v, 3)})
if high_corr:
    print("Highly correlated feature pairs (|r| > 0.6):")
    print(pd.DataFrame(high_corr).sort_values('correlation', key=abs, ascending=False).to_string(index=False))